# Phase 2-alpha: evaluate gpt-oss-120b as a zero-shot parser on gold corporaRuns the 5 in-domain test sets through gpt-oss-120b via Cerebras as a zero-shot parser. Produces the Phase 2-alpha column in Appendix B.Requires a `CEREBRAS_API_KEY` in the environment.

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate

export CEREBRAS_API_KEY=<CEREBRAS_API_KEY>

# Wipe the empty predictions so they get re-done
rm -rf phase2_data/phase2alpha_indomain_preds/

nohup python3 -u <<'PY' > eval_logs/phase2alpha_parser_eval.log 2>&1 &
import sys, json, time, os
sys.path.insert(0, '.')
from pathlib import Path
from src.phase2.config import TeacherConfig
from src.phase2.teacher import build_teacher
from src.phase2.dataset import read_jsonl
from src.phase2.evaluate import evaluate_corpus

cfg = TeacherConfig(
    backend="cerebras",
    model="gpt-oss-120b",
    max_input_chars=2000,
    max_output_tokens=2048,
    rpm_cap=30,
    max_requests_per_day=1000,
    retries_per_call=3,
)
teacher = build_teacher(cfg)
print(f"[eval] teacher: {teacher.name} ({cfg.model})", flush=True)

CAP = 50
PRED_DIR = Path("phase2_data/phase2alpha_indomain_preds")
PRED_DIR.mkdir(parents=True, exist_ok=True)
summary = {}

for dom in ("microtext", "cdcp", "abstrct", "perspectrum"):
    recs = read_jsonl(f"phase2_data/unified/{dom}_test.jsonl")[:CAP]
    if not recs: continue

    pred_path = PRED_DIR / f"{dom}.jsonl"
    done = {}
    if pred_path.exists():
        with open(pred_path) as f:
            for i, line in enumerate(f):
                try: done[i] = json.loads(line)["prediction"]
                except: pass
    print(f"[{dom}] {len(recs)} records, {len(done)} already done", flush=True)

    preds = [None] * len(recs)
    for i, p in done.items():
        if i < len(preds): preds[i] = p

    t0 = time.time()
    n_new = 0
    with open(pred_path, "a") as fout:
        for i, rec in enumerate(recs):
            if preds[i] is not None: continue
            pred, reasoning = teacher.annotate(rec["input"], source=dom)
            preds[i] = pred
            fout.write(json.dumps({"record_index": i, "prediction": pred,
                                   "reasoning_marker": reasoning[:120]},
                                  ensure_ascii=False) + "\n")
            fout.flush(); os.fsync(fout.fileno())
            n_new += 1
            if (i+1) % 10 == 0:
                elapsed = time.time() - t0
                rate = n_new / max(elapsed, 1e-6)
                eta = (len(recs) - (i+1)) / max(rate, 1e-6) / 60
                # Flag if returning the empty fallback
                empty_so_far = sum(1 for p in preds[:i+1]
                                   if p and len(p['claim_components'])==0
                                   and len(p['premise_components'])==0)
                print(f"  {dom} {i+1}/{len(recs)} ({elapsed:.0f}s, "
                      f"ETA {eta:.1f}min, empty_so_far={empty_so_far})",
                      flush=True)

    m = evaluate_corpus(recs, preds, threshold=0.5)
    n_empty = sum(1 for p in preds
                  if len(p['claim_components'])==0
                  and len(p['premise_components'])==0)
    print(f"\n── {dom} ── n={len(recs)} empty={n_empty} "
          f"({100*n_empty/len(recs):.0f}%) ({time.time()-t0:.0f}s)",
          flush=True)
    print(f"   comp-F1={m['macro_component_f1']:.3f} "
          f"rel-F1={m['relation_f1']['f1']:.3f} "
          f"claim-F1={m['component_f1']['claim']['f1']:.3f} "
          f"premise-F1={m['component_f1']['premise']['f1']:.3f}\n",
          flush=True)
    summary[dom] = {
        "n": len(recs), "empty": n_empty,
        "comp_f1": m["macro_component_f1"],
        "rel_f1": m["relation_f1"]["f1"],
        "claim_f1": m["component_f1"]["claim"]["f1"],
        "premise_f1": m["component_f1"]["premise"]["f1"],
        "citation_f1": m["component_f1"]["citation"]["f1"],
    }

print("\n=== SUMMARY ===", flush=True)
for dom, s in summary.items():
    print(f"  {dom:14s} comp-F1={s['comp_f1']:.3f} "
          f"rel-F1={s['rel_f1']:.3f} empty={s['empty']}/{s['n']}", flush=True)
if summary:
    avg = sum(s['comp_f1'] for s in summary.values()) / len(summary)
    print(f"  {'average':14s} comp-F1={avg:.3f}", flush=True)

Path("eval_logs/phase2alpha_indomain_eval.json").write_text(json.dumps({
    "teacher": "gpt-oss-120b via Cerebras (single key)",
    "method": "zero-shot parser on in-domain held-out test sets",
    "cap_per_domain": CAP,
    "per_domain": summary,
    "average_comp_f1": sum(s['comp_f1'] for s in summary.values())/max(len(summary),1),
}, indent=2))
PY
echo "PID: $!"
sleep 10
tail -15 eval_logs/phase2alpha_parser_eval.log

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate

export CEREBRAS_API_KEY=<CEREBRAS_API_KEY>

nohup python3 -u <<'PY' > eval_logs/phase2alpha_parser_eval.log 2>&1 &
import sys, json, time, os
sys.path.insert(0, '.')
from pathlib import Path
from src.phase2.config import TeacherConfig
from src.phase2.teacher import build_teacher
from src.phase2.dataset import read_jsonl
from src.phase2.evaluate import evaluate_corpus

cfg = TeacherConfig(
    backend="cerebras", model="gpt-oss-120b",
    max_input_chars=2000, max_output_tokens=2048,
    rpm_cap=30, max_requests_per_day=1000, retries_per_call=3,
)
teacher = build_teacher(cfg)
print(f"[eval] teacher: {teacher.name} ({cfg.model})", flush=True)

CAP = 50
PRED_DIR = Path("phase2_data/phase2alpha_indomain_preds")
PRED_DIR.mkdir(parents=True, exist_ok=True)
summary = {}

for dom in ("microtext", "cdcp", "abstrct", "perspectrum"):
    recs = read_jsonl(f"phase2_data/unified/{dom}_test.jsonl")[:CAP]
    if not recs: continue
    pred_path = PRED_DIR / f"{dom}.jsonl"
    done = {}
    if pred_path.exists():
        with open(pred_path) as f:
            for i, line in enumerate(f):
                try: done[i] = json.loads(line)["prediction"]
                except: pass
    print(f"[{dom}] {len(recs)} records, {len(done)} already done", flush=True)
    preds = [None] * len(recs)
    for i, p in done.items():
        if i < len(preds): preds[i] = p
    t0 = time.time(); n_new = 0
    with open(pred_path, "a") as fout:
        for i, rec in enumerate(recs):
            if preds[i] is not None: continue
            pred, reasoning = teacher.annotate(rec["input"], source=dom)
            preds[i] = pred
            fout.write(json.dumps({"record_index": i, "prediction": pred,
                                   "reasoning_marker": reasoning[:120]},
                                  ensure_ascii=False) + "\n")
            fout.flush(); os.fsync(fout.fileno())
            n_new += 1
            if (i+1) % 10 == 0:
                elapsed = time.time()-t0
                rate = n_new / max(elapsed, 1e-6)
                eta = (len(recs)-(i+1))/max(rate,1e-6)/60
                empty_so_far = sum(1 for p in preds[:i+1]
                                   if p and len(p['claim_components'])==0
                                   and len(p['premise_components'])==0)
                print(f"  {dom} {i+1}/{len(recs)} ({elapsed:.0f}s, "
                      f"ETA {eta:.1f}min, empty_so_far={empty_so_far})", flush=True)
    m = evaluate_corpus(recs, preds, threshold=0.5)
    n_empty = sum(1 for p in preds
                  if len(p['claim_components'])==0
                  and len(p['premise_components'])==0)
    print(f"\n── {dom} ── n={len(recs)} empty={n_empty} "
          f"({100*n_empty/len(recs):.0f}%)", flush=True)
    print(f"   comp-F1={m['macro_component_f1']:.3f} "
          f"rel-F1={m['relation_f1']['f1']:.3f} "
          f"claim-F1={m['component_f1']['claim']['f1']:.3f} "
          f"premise-F1={m['component_f1']['premise']['f1']:.3f}\n", flush=True)
    summary[dom] = {
        "n": len(recs), "empty": n_empty,
        "comp_f1": m["macro_component_f1"],
        "rel_f1": m["relation_f1"]["f1"],
        "claim_f1": m["component_f1"]["claim"]["f1"],
        "premise_f1": m["component_f1"]["premise"]["f1"],
        "citation_f1": m["component_f1"]["citation"]["f1"],
    }

print("\n=== SUMMARY ===", flush=True)
for dom, s in summary.items():
    print(f"  {dom:14s} comp-F1={s['comp_f1']:.3f} "
          f"rel-F1={s['rel_f1']:.3f} empty={s['empty']}/{s['n']}", flush=True)
if summary:
    avg = sum(s['comp_f1'] for s in summary.values())/len(summary)
    print(f"  {'average':14s} comp-F1={avg:.3f}", flush=True)
Path("eval_logs/phase2alpha_indomain_eval.json").write_text(json.dumps({
    "teacher": "gpt-oss-120b via Cerebras (single key)",
    "method": "zero-shot parser on in-domain held-out test sets",
    "cap_per_domain": CAP, "per_domain": summary,
    "average_comp_f1": sum(s['comp_f1'] for s in summary.values())/max(len(summary),1),
}, indent=2))
PY
echo "PID: $!"
sleep 10
tail -15 eval_logs/phase2alpha_parser_eval.log